In [ ]:
# Import dependencies

import glob
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_PI_Kersting/am84fuxo/LlavaGuard')
import nest_asyncio
nest_asyncio.apply()
import pandas as pd
import os
import json
from llavaguard.taxonomy.PEGI.PEGI_Graph import remove_numbers_from_categories, get_policy_assessment, policy_graph, policy_graph_to_safety_policy_v2, get_rating
from llavaguard.taxonomy.PEGI.PEGI_Graph import get_content_categories, get_content_categories_with_examples, get_content_categories_with_numbers, get_policy_intro, get_safety_categories, policy_graph, policy_graph_to_safety_policy
from llavaguard_config import local_image_dirs, local_data_dir
from pegiguard.data.dataset_utils import parse_labels, labels_to_string


In [ ]:
# Extract accepted annotation categories

# Extract category assessments from majority vote.

data = f'/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/annotations/auto_generated_annotations/v0_2/majority_vote.csv'
output_dir = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/annotations/auto_generated_annotations/v0_2/'
df = pd.read_csv(data)
df = df[df['voting_mechanism'] == 'majority_vote']
categories = df.columns[3:]
extracted_data = []
for index, row in df.iterrows():
    
    for category in categories:
        if row[category] == 1:
            extracted_data.append({
                'image_name': row['sample_id'],
                'file_path': row['im_path'],
                'category': category,
            })
result_df = pd.DataFrame(extracted_data)
result_df.to_csv(f'{output_dir}/extracted_categories_numbers.csv', index=False)


In [ ]:
# Correct legacy image paths

# Change image paths to correct ones.

import pandas as pd

dir = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/annotations/auto_generated_annotations/v0_2/'

df = pd.read_csv(f'{dir}/extracted_categories.csv', header=None, names=["image_id", "path", "category"])


old_prefix = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/lhelff/ds/LlavaGuard/data/"
new_prefix = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/"


df["path"] = df["path"].str.replace(old_prefix, new_prefix, regex=False)

df.to_csv(f'{dir}/extracted_categories.csv', index=False, header=False)


In [ ]:
# Load a local model

from transformers import AutoProcessor, Llama4ForConditionalGeneration, AutoTokenizer
import torch
from PIL import Image


model_id = "meta-llama/Llama-4-Scout-17B-16E-Instruct"

processor = AutoProcessor.from_pretrained(model_id)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id,
    attn_implementation="flex_attention",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)


In [ ]:
# Inspect text-only generation

# Testing llama4 with transformers, only text input.
import time


import transformers


from transformers import AutoTokenizer


from transformers import AutoProcessor


from transformers import Llama4ForConditionalGeneration


# ###.

model_id = "meta-llama/Llama-4-Scout-17B-16E-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id, device_map="auto", 
    torch_dtype=torch.bfloat16,
)

outputs = model.generate(**inputs.to(model.device), max_new_tokens=100)
outputs = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[-1] :])
print(outputs[0])


In [ ]:
# Generate category rationales

# Generating rationales.
import os
import argparse
import glob
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/LlavaGuard')
import torch
print("before transformers")
from transformers import AutoProcessor, Llama4ForConditionalGeneration
print("after transformers")
from PIL import Image
from llavaguard_config import local_image_dirs, local_data_dir

import pandas as pd
import json

# Loading model.

model_id = "meta-llama/Llama-4-Scout-17B-16E-Instruct"

processor = AutoProcessor.from_pretrained(model_id)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id,
    attn_implementation="flex_attention",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Generating rationales for all the images.


processor = AutoProcessor.from_pretrained(model_id)
local_data_dir = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo" 
csv_path      = f"{local_data_dir}/data/annotations/auto_generated_annotations/v0_2/extracted_categories_stripped.csv"
output_path   = f"{local_data_dir}/data/PEGI-LlavaGuard-DS/v2/all_data.json"
model_id      = "meta-llama/Llama-4-Scout-17B-16E-Instruct"


df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows from:\n  {csv_path}")

df = df.head(5)
total = len(df)
all_outputs = []
checkpoint_every = 2

for idx, row in df.iterrows():
    sample_id = row["image_name"]
    print(row["file_path"])
    img_path  = row["file_path"]
    category  = row["category"]

    prompt = (
        f"Describe the image and how it relates to the following PEGI rating safety-category - {category} - in 2 sentences."
    )
    print(f"\n[Sample {idx + 1} / {len(df)}]")
    
    image = Image.open(img_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(**inputs, max_new_tokens=200)

    start = inputs["input_ids"].shape[-1]
    rationale_text = processor.batch_decode(outputs[:, start:])[0].strip()
    
    

    all_outputs.append({
        "image_name": sample_id,
        "image_path": img_path,
        "category":   category,
        "rationale":  rationale_text
    })
    if (idx + 1) % checkpoint_every == 0 or idx + 1 == total:
        parent_dir = os.path.dirname(output_path)
        os.makedirs(parent_dir, exist_ok=True)
        with open(output_path, "w") as f:
            json.dump(all_outputs, f, indent=4)
        print(f"  - [Checkpoint] Wrote {idx+1} / {total} entries to {output_path}", flush=True)


# Os.makedirs(parent_dir, exist_ok=True).

# With open(output_path, "w") as f.
# Json.dump(all_outputs, f, indent=4).

print(f"\n Written {len(all_outputs)} entries to:\n {output_path}")


In [ ]:
# Check rating compliance

def assessment(compliant_rating, curr_rating):
    if (curr_rating <= compliant_rating):
        return "Compliant"
    else:
        return "Review Needed"


In [ ]:
# Export legacy instruction data

# Transform dataset to LLama-factory Format.
import json
import os
import csv
import json

rationale_path = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGI-LlavaGuard-DS/rationales.json'
output_path   = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGI-LlavaGuard-DS/all_data_v3.json'
maj_vote = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/annotations/auto_generated_annotations/v0_2/extracted_categories_numbers.csv'

mapping = {}
with open(maj_vote, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)
    for image_name, img_path_csv, full_cat in reader:
        number, cat_name = full_cat.split(". ", 1)
        mapping[(image_name, cat_name)] = number


with open(rationale_path, "r", encoding="utf-8") as f:
    original = json.load(f)

compliant_rating = 18
transformed = []
for item in original:
    image_id = item["image_name"]
    raw_cat  = item["category"]
    number   = mapping.get((image_id, raw_cat))

    if number:
        cat_with_number = f"{number}. {raw_cat}"
    else:
        cat_with_number = raw_cat
        print("failed to get a number")
    
    rating = get_rating(raw_cat)
    
    clean_rationale = item["rationale"].split("<|eot|>")[0].strip()

    gpt_value = json.dumps({
        "rationale": clean_rationale,
        "category":  cat_with_number,
        "PEGI-rating": str(rating)
    }, indent=4)

    transformed.append({
        "id":          image_id,
        "image":       item["image_path"],
        "category":    cat_with_number,
        "PEGI-rating": str(rating),
        "conversations": [
            {"from": "human", "value": policy_graph_to_safety_policy(compliant_rating)},
            {"from": "gpt",   "value": gpt_value}
        ]
    })


with open(output_path, "w", encoding="utf-8") as f:
    json.dump(transformed, f, indent=4, ensure_ascii=False)

print(f"Written {len(transformed)} entries to {output_path}")


print(json.dumps(transformed[0], indent=4, ensure_ascii=False))


In [ ]:
# Inspect exported data

import json
from pprint import pprint

output_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGI-LlavaGuard-DS/all_data.json"


with open(output_path, "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} records\n")

for i, rec in enumerate(data[:5]):
    print(f"Record {i} raw:")
    pprint(rec)
    print("Keys:", list(rec.keys()))
    print("-" * 40)


In [ ]:
# Configure safety rationale generation

import os
import json
import argparse
from openai import OpenAI
from PIL import Image
import random
import textwrap
import sys
import logging
from pathlib import Path

sys.path.append('/pfss/mlde/workspaces/mlde_wsp_PI_Kersting/am84fuxo/LlavaGuard')

from llavaguard.taxonomy.PEGI.PEGI_Graph import (
    policy_graph, policy_graph_to_safety_policy, get_max_pegi_score, policy_graph_to_rationale
)
from llavaguard_config import local_data_dir
from pegiguard.data.dataset_utils import pil_to_data_url

# Constants.
MODEL_ID = "Qwen/Qwen3.5-35B-A3B"
CHECK_POINT = 50

# Configure logging.
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


In [ ]:
# Generate age-dependent safety rationales

def get_user_age(idx: int) -> int:
    """
    Returns a random user age that is allowed for a certain PEGI score.

    PEGI levels by index:
        0 -> PEGI 3 allowed ages 3-6
        1 -> PEGI 7 allowed ages 7-11
        2 -> PEGI 12 allowed ages 12-15
        3 -> PEGI 16 allowed ages 16-17
        4 -> PEGI 18 allowed ages 18+
    """
    age_ranges = [
        (3, 6),      # Idx 0: 3–6.
        (7, 11),     # Idx 1: 7–11.
        (12, 15),    # Idx 2: 12–15.
        (16, 17),    # Idx 3: 16–17.
        (18, None)   # Idx 4: 18+.
    ]

    if idx < 0 or idx >= len(age_ranges):
        raise ValueError(f"idx must be between 0 and {len(age_ranges) - 1}")

    low, high = age_ranges[idx]

    if high is None:
        # PEGI 18 = any adult age (e.g. 18–99).
        return random.randint(low, 99)
    else:
        return random.randint(low, high)


def get_pegi_rating(index: int) -> int:
    """Return PEGI rating number given an index (0–5)."""
    mapping = {
        0: 3,
        1: 7,
        2: 12,
        3: 16,
        4: 18
    }
    if index not in mapping:
        raise ValueError(f"Invalid index {index}. Must be between 0 and 5.")
    return mapping[index]

def get_rationale(client: OpenAI, img_path: str, assessment: str, categories: str, subcategories: list, pegi_score: int) -> str:
    """
    Generate a rationale for the image assessment using the OpenAI API.

    Args:
        client: OpenAI client instance
        img_path: Path to the image file
        assessment: "Safe" or "Unsafe"
        categories: Category string
        subcategories: List of subcategories
        pegi_score: PEGI score number

    Returns:
        JSON string with categories, subcategories, rationale, and assessment
    """
    try:
        image = Image.open(img_path).convert("RGB")
        data_url = pil_to_data_url(image)
    except Exception as e:
        logger.error(f"Error loading image {img_path}: {e}")
        raise

    if subcategories:
        cats_str = "\n".join(f"- {c}" for c in subcategories)
    else:
        cats_str = "NA: None applying"

    rationale_policy = policy_graph_to_rationale(pegi_score)

    base_prompt = f"""Consider the following policy for PEGI {pegi_score}:
{rationale_policy}

You are a content safety analyst writing rationales for a labelled dataset.
The provided labels and assessment are authoritative and must not be questioned.

Write a short rationale using only visible evidence in the image.

Instructions:
- Start with 1–2 sentences describing the image.
- Then, in up to 4 sentences total, explain how the image relates to these categories (only include applicable ones, do not mention non-applicable ones):
{cats_str}

Constraints:
- Do not contradict the given categories or assessment.
- Do not introduce new categories.
- Use category names exactly as written.
- Do not mention uncertainty or alternative interpretations.
- No section labels or lists.
- Keep it concise.
"""
    # Build prompt based on assessment and categories.
    if assessment == "Unsafe":
        prompt = base_prompt + f"""
Conclude by stating which of the listed categories violate the PEGI {pegi_score} policy (i.e., are prohibited).
"""
    elif subcategories:
        prompt = base_prompt + f"""
Conclude by explaining why none of these categories violate the PEGI {pegi_score} policy and why the image is safe for users aged {pegi_score}+.
"""
    else:
        prompt = f"""Consider the following policy for PEGI {pegi_score}:
{rationale_policy}

You are a content safety analyst writing rationales for a labelled dataset.
The provided label and assessment are authoritative.

Describe the image in 1–2 sentences, then explain briefly why the absence of relevant safety categories makes it compliant with PEGI {pegi_score}.

Constraints:
- Do not contradict the given categories or assessment.
- Do not introduce categories.
- No section labels or lists.
- Keep it concise.
"""

    messages = [
        {"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": data_url}},
        ]},
    ]

    try:
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
        )
        response = resp.choices[0].message.content.strip()
    except Exception as e:
        logger.error(f"Error calling OpenAI API for image {img_path}: {e}")
        raise

    result = {
        "categories": categories,
        "subcategories": labels_to_string(subcategories),
        "rationale": response,
        "assessment": assessment,
    }
    return json.dumps(result, indent=4, ensure_ascii=False)


In [ ]:
# Connect to the inference server

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")


In [ ]:
# Inspect a generated rationale

import matplotlib.pyplot as plt
from PIL import Image
import json
import os

# Load the dataset.
dataset_path = "/pfss/mlde/workspaces/mlde_wsp_PI_Kersting/am84fuxo/data/datasets/PEGI-Guard-DS/v4/all_data.json"

with open(dataset_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# Set the entry index (n-th entry, 0-indexed).
n = 15  # Change this to desired index (0 for first entry, 1 for second, etc.).

if n >= len(dataset):
    print(f"Error: Index {n} out of range. Dataset has {len(dataset)} entries.")
else:
    sample = dataset[n]
    
    print(f"\n{'='*80}")
    print(f"ENTRY #{n}")
    print(f"{'='*80}\n")
    
    # Print original sample info.
    img_id = sample.get("id")
    img_path = sample.get("image")  # Changed from "image_path" to "image".
    categories = sample.get("categories")
    subcategories_str = sample.get("subcategories")
    
    print(f"Image ID: {img_id}")
    print(f"Image Path: {img_path}")
    print(f"Categories: {categories}")
    print(f"Subcategories: {subcategories_str}")
    
    # Parse subcategories from string format.
    # E.g., "1. Educational Content, 24. Highly Disturbing Content, 5. Violent Acts (animals), 9. Gross Violence".
    subcategories_list = parse_labels(subcategories_str)

    print(f"Parsed Subcategories List: {subcategories_list}\n")
    
    # Get PEGI score and generate rationale.
    from llavaguard.taxonomy.PEGI.PEGI_Graph import get_max_pegi_score, policy_graph_to_safety_policy
    
    pegi_score = get_max_pegi_score(subcategories_list)
    user_pegi = pegi_score  # Use the max PEGI score.
    
    assessment = "Safe" if user_pegi >= pegi_score else "Unsafe"
    
    print(f"PEGI Score: {pegi_score}")
    print(f"Assessment: {assessment}\n")
    
    # Generate rationale.
    try:
        print("Generating rationale...")
        gpt_value = get_rationale(client, img_path, assessment, categories, subcategories_list, user_pegi)
        rationale_dict = json.loads(gpt_value)
        
        print(f"\n{'='*80}")
        print("GENERATED RATIONALE:")
        print(f"{'='*80}")
        print(json.dumps(rationale_dict, indent=2, ensure_ascii=False))
        
    except Exception as e:
        print(f"Error generating rationale: {e}")
        import traceback
        traceback.print_exc()
    
    # Display the image.
    try:
        if img_path and os.path.exists(img_path):
            image = Image.open(img_path).convert("RGB")
            
            plt.figure(figsize=(12, 8))
            plt.imshow(image)
            plt.axis('off')
            plt.title(f"Image ID: {img_id}")
            plt.tight_layout()
            plt.show()
            
            print(f"\nImage displayed successfully from: {img_path}")
        else:
            print(f"Warning: Image not found at path: {img_path}")
    except Exception as e:
        print(f"Error displaying image: {e}")
    
    # Print all sample data.
    print(f"\n{'='*80}")
    print("COMPLETE ENTRY DATA:")
    print(f"{'='*80}")
    print(json.dumps(sample, indent=2, ensure_ascii=False))
    print(f"{'='*80}\n")
